# 🏥 Fine-tuning Phi-3.5 pour le domaine médical (QLoRA)

Ce notebook fine-tune le modèle **Phi-3.5-mini-instruct** avec notre dataset nettoyé.

> ⚠️ **Pré-requis** : Runtime > Change runtime type > **T4 GPU**

> ⚠️ **Après la cellule 1 (pip install)**, faites **Runtime > Restart session** puis relancez à partir de la **cellule 2**.

In [ ]:
# ============================
# CELLULE 1 : Installation
# ============================
!pip install -q -U transformers accelerate datasets
!pip install -q --force-reinstall bitsandbytes>=0.46.1
!pip install -q -U peft>=0.15.0
!pip install -q -U trl>=0.18.0
!pip uninstall -y torchvision torchaudio
import os
print("✅ Installation terminée ! Le noyau va redémarrer automatiquement pour charger la bonne version...")
os.kill(os.getpid(), 9)


In [ ]:
# ============================
# CELLULE 2 : Chargement du modèle en 4-bit (QLoRA)
# ============================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

model_id = "microsoft/Phi-3.5-mini-instruct"

# Quantization 4-bit pour tenir en mémoire sur T4 (16 Go)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("📝 Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("🧠 Chargement du modèle en 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# Config LoRA (on ne l'applique PAS manuellement, SFTTrainer s'en charge)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(f"✅ Modèle chargé ! Device: {model.device}")

In [ ]:
# ============================
# CELLULE 3 : Chargement et préparation du dataset
# ============================
# ⚠️ Uploadez 'test_dataset_16000_clean.json' via le panneau fichiers à gauche
from datasets import load_dataset

dataset = load_dataset("json", data_files="test_dataset_16000_clean.json", split="train")

def format_to_phi(example):
    instruction = example.get("instruction", "")
    inp = example.get("input", "")
    output = example.get("output", "")
    prompt = f"{instruction} {inp}".strip()
    example["text"] = f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n{output}<|end|>"
    return example

dataset = dataset.map(format_to_phi)
print(f"✅ Dataset prêt : {len(dataset)} exemples")
print(f"📝 Exemple : {dataset[0]['text'][:200]}...")

In [ ]:
# ============================
# CELLULE 4 : Lancement du fine-tuning
# ============================
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./outputs",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    max_steps=500,
    learning_rate=2e-4,
    fp16=False,
    logging_steps=10,
    save_steps=100,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

print("🚀 Début de l'entraînement...")
result = trainer.train()
print(f"\n✅ Entraînement terminé !")
print(f"📉 Loss finale : {result.training_loss:.4f}")